# Publicar Mobilidade Urbana no Portal da Prefeitura — v2
1. Copia do Portal Engemap (camadas públicas, sem login) para FGDBs locais: **Linhas de ônibus** (74), **Pontos_bus_temp** (1860) e **CSV_Master_de_Horarios_v2** (1369).
2. Publica cada uma como **camada hosted separada** no Portal da Prefeitura, na pasta **Mobilidade_Urbana**, compartilhada como pública.
3. Cria o web map **Mobilidade Urbana** na Prefeitura: Bairros e Eixos (já existentes), Linhas com pop-up de horários (Arcade, pelo id_rota), Pontos e uma cópia dos Pontos com mapa de calor (desligada).
4. Grava os IDs gerados (para o ExB) e imprime os links.

**Seguro para rodar de novo:** se um item já existir na pasta, ele é reaproveitado (não duplica nem sobrescreve).
Saída: `Documents\Saidas_Notebooks\Publicar_Mobilidade_Apucarana_v2.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, sys, json, datetime, re, shutil, zipfile, copy, unicodedata, traceback

NB_NOME   = "Publicar_Mobilidade_Apucarana_v2"
URL_ENG   = "https://gisserver.engemap.com.br/portal"
URL_APUC  = "https://geo.apucarana.pr.gov.br/portal"
PASTA_PORTAL = "Mobilidade_Urbana"
COMPARTILHAR_PUBLICO = True

# Origem (Portal Engemap) — itens públicos
SRC = {
    "linhas":   {"item": "56893159744147d89d3b83f21484b026", "sub": 0, "tabela": False,
                 "titulo": "Linhas de Ônibus - Mobilidade Urbana", "servico": "Mobilidade_Linhas_Onibus",
                 "tags": "mobilidade, ônibus, linhas, transporte coletivo, Apucarana, GeoAPUC",
                 "resumo": "Itinerários das linhas de ônibus de Apucarana (74 rotas, campo id_rota liga aos horários)."},
    "pontos":   {"item": "446aa70926194cf6821f34ac5cae9235", "sub": 0, "tabela": False,
                 "titulo": "Pontos de Ônibus - Mobilidade Urbana", "servico": "Mobilidade_Pontos_Onibus",
                 "tags": "mobilidade, ônibus, pontos, paradas, transporte coletivo, Apucarana, GeoAPUC",
                 "resumo": "Pontos de ônibus de Apucarana encaminhados pelo setor de transportes."},
    "horarios": {"item": "2d347f0486bf4cd0858ae7fc06b3cd17", "sub": 0, "tabela": True,
                 "titulo": "Horários de Ônibus - Mobilidade Urbana", "servico": "Mobilidade_Horarios_Onibus",
                 "tags": "mobilidade, ônibus, horários, transporte coletivo, Apucarana, GeoAPUC",
                 "resumo": "Quadro de horários das linhas de ônibus de Apucarana (liga às linhas pelo campo id_rota)."},
}
# Já existentes no Portal da Prefeitura (Feature Service)
APUC_BAIRROS = "155226d377e0412a8e8eb0e01f66889b"
APUC_EIXOS   = "74dbea774e4e44ddbb8b5c821c8d9b8d"
WEBMAP_TITULO = "Mobilidade Urbana"
# Web maps do protótipo (para copiar simbologia)
WM_ENG_PRINCIPAL = "9657691b50a0421d9fcd6e1671d8e1c3"
WM_ENG_CALOR     = "ccc02c09c798462c9d2d1b983321a2a6"

home = os.path.expanduser("~")
_cands = [os.path.join(home, "Documents", "Saidas_Notebooks"),
          os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks"),
          os.path.join(home, "OneDrive", "Documentos", "Saidas_Notebooks"),
          os.path.join(home, "Documentos", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _cands if os.path.isdir(c)), _cands[0])
os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_INV = os.path.join(PASTA_SAIDA, "Inventario_Mobilidade_Engemap_x_Apucarana_v1_json")
PASTA_TRAB = os.path.join(PASTA_SAIDA, NB_NOME + "_trabalho")
os.makedirs(PASTA_TRAB, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f:
        f.write(s + "\n")

LINKS = []
log("▶ CÉLULA 1 — Configuração: iniciando...")
log("  Saída (.txt):", ARQ_TXT)
log("  Pasta de trabalho (FGDBs/zips):", PASTA_TRAB)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Conexões =====
log("\n▶ CÉLULA 2 — Conexões com os portais: iniciando...")
import getpass, arcpy
from arcgis.gis import GIS
ATIVO = ""
try:
    ATIVO = arcpy.GetActivePortalURL() or ""
except Exception:
    pass
log("  Portal ativo no ArcGIS Pro:", ATIVO)
if "apucarana" in ATIVO.lower():
    gis_apuc = GIS("pro")
else:
    log("  O portal ativo não é o da Prefeitura. Digite a senha do idepplan1.")
    gis_apuc = GIS(URL_APUC, "idepplan1", getpass.getpass("Senha idepplan1 @ Apucarana: "))
ME = gis_apuc.users.me
log("  Prefeitura: conectado como", ME.username, "|", gis_apuc.url)
gis_eng = GIS(URL_ENG)   # anônimo — as camadas de origem são públicas
log("  Engemap: acesso anônimo (camadas públicas)")
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Copiar da Engemap para FGDBs locais =====
log("\n▶ CÉLULA 3 — Cópia das camadas da Engemap para FGDB: iniciando...")
arcpy.env.overwriteOutput = True
for chave, s in SRC.items():
    it = gis_eng.content.get(s["item"])
    if it is None:
        raise RuntimeError(f"Item de origem {s['item']} ({chave}) não encontrado na Engemap.")
    url = it.url.rstrip("/") + f"/{s['sub']}"
    s["url_origem"] = url
    gdb_dir = os.path.join(PASTA_TRAB, s["servico"] + ".gdb")
    if os.path.isdir(gdb_dir):
        try: arcpy.management.Delete(gdb_dir)
        except Exception: shutil.rmtree(gdb_dir, ignore_errors=True)
    arcpy.management.CreateFileGDB(PASTA_TRAB, s["servico"] + ".gdb")
    saida = os.path.join(gdb_dir, s["servico"])
    log(f"  · {chave}: copiando {url}")
    if s["tabela"]:
        arcpy.conversion.ExportTable(url, saida)
    else:
        arcpy.conversion.ExportFeatures(url, saida)
    n = int(arcpy.management.GetCount(saida)[0])
    campos = [f.name for f in arcpy.ListFields(saida)]
    s["gdb"], s["n_origem"] = gdb_dir, n
    log(f"    → {n} registros | campos: {', '.join(campos)}")
arcpy.management.ClearWorkspaceCache()
# compactar cada FGDB (sem arquivos .lock)
for chave, s in SRC.items():
    z = os.path.join(PASTA_TRAB, s["servico"] + ".zip")
    if os.path.exists(z): os.remove(z)
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        base = os.path.basename(s["gdb"])
        for raiz, _, arqs in os.walk(s["gdb"]):
            for a in arqs:
                if a.lower().endswith(".lock"):
                    continue
                full = os.path.join(raiz, a)
                zf.write(full, os.path.join(base, os.path.relpath(full, s["gdb"])))
    s["zip"] = z
    log(f"  · zip pronto: {z}")
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Publicar no Portal da Prefeitura (pasta Mobilidade_Urbana) =====
log("\n▶ CÉLULA 4 — Publicação no Portal da Prefeitura: iniciando...")
# ---- compatível com arcgis 2.3 (dict) e 2.4+ (objeto Folder) ----
def _nome_pasta(f):
    if isinstance(f, dict):
        return f.get("title")
    return getattr(f, "name", None) or getattr(f, "title", None)

def pasta_obj():
    for f in (ME.folders or []):
        if _nome_pasta(f) == PASTA_PORTAL:
            return f
    return None

if pasta_obj() is None:
    try:
        gis_apuc.content.folders.create(PASTA_PORTAL)
    except Exception:
        gis_apuc.content.create_folder(PASTA_PORTAL)
    ME = gis_apuc.users.get(ME.username)   # recarrega a lista de pastas
    log("  Pasta criada:", PASTA_PORTAL)
else:
    log("  Pasta já existe:", PASTA_PORTAL)

def itens_na_pasta():
    try:
        return list(ME.items(folder=PASTA_PORTAL, max_items=500))
    except Exception:
        f = pasta_obj()
        return list(f.list()) if f is not None and hasattr(f, "list") else []

def adicionar(props, data=None):
    """Adiciona item na pasta — tenta a API antiga e, se falhar, a nova (2.4+)."""
    try:
        if data:
            return gis_apuc.content.add(item_properties=props, data=data, folder=PASTA_PORTAL)
        return gis_apuc.content.add(item_properties=props, folder=PASTA_PORTAL)
    except Exception as e1:
        from arcgis.gis import ItemProperties, ItemTypeEnum
        tipo = {"File Geodatabase": ItemTypeEnum.FILE_GEODATABASE, "Web Map": ItemTypeEnum.WEB_MAP}[props["type"]]
        ip = ItemProperties(title=props["title"], item_type=tipo, tags=props.get("tags"),
                            snippet=props.get("snippet"), description=props.get("description"),
                            text=props.get("text"))
        f = pasta_obj() or gis_apuc.content.folders.get()
        job = f.add(item_properties=ip, file=data) if data else f.add(item_properties=ip)
        return job.result() if hasattr(job, "result") else job

def compartilhar(item):
    if not COMPARTILHAR_PUBLICO: return
    try:
        item.sharing.sharing_level = "EVERYONE"
    except Exception:
        try: item.share(everyone=True)
        except Exception as e: log("    ! não consegui compartilhar como público:", e)

for chave, s in SRC.items():
    existente = [i for i in itens_na_pasta() if i.type == "Feature Service" and i.title == s["titulo"]]
    if existente:
        fs = existente[0]
        log(f"  · {chave}: já publicado — reaproveitando {fs.id}")
    else:
        props = {"title": s["titulo"], "type": "File Geodatabase", "tags": s["tags"],
                 "snippet": s["resumo"], "description": s["resumo"]}
        gdb_item = [i for i in itens_na_pasta() if i.type == "File Geodatabase" and i.title == s["titulo"]]
        gdb_item = gdb_item[0] if gdb_item else adicionar(props, s["zip"])
        log(f"  · {chave}: FGDB enviado ({gdb_item.id}) — publicando como '{s['servico']}'...")
        fs = gdb_item.publish(publish_parameters={"name": s["servico"]}, file_type="fileGeodatabase")
        try:
            fs.update(item_properties={"title": s["titulo"], "tags": s["tags"], "snippet": s["resumo"]})
            if fs.ownerFolder is None:
                fs.move(PASTA_PORTAL)
        except Exception as e:
            log("    ! ajuste de propriedades:", e)
        log(f"    → publicado: {fs.id}")
    compartilhar(fs)
    s["novo_id"], s["novo_url"] = fs.id, fs.url.rstrip("/") + f"/{0}"
    from arcgis.features import FeatureLayer, Table
    lyr = (fs.tables[0] if s["tabela"] else fs.layers[0])
    n = lyr.query(where="1=1", return_count_only=True)
    s["n_destino"] = n
    ok = "OK" if n == s["n_origem"] else "DIFERENTE — conferir!"
    log(f"    contagem: origem={s['n_origem']} × Prefeitura={n} → {ok}")
    LINKS.append((f"Item — {s['titulo']}", f"{URL_APUC}/home/item.html?id={fs.id}"))
log("✔ CÉLULA 4 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 5 — Web map "Mobilidade Urbana" na Prefeitura =====
log("\n▶ CÉLULA 5 — Criação do web map: iniciando...")

def ler_backup(iid):
    p = os.path.join(PASTA_INV, f"webmap_engemap_{iid}.json")
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    try:
        return gis_eng.content.get(iid).get_data()
    except Exception:
        return {}

wm_p = ler_backup(WM_ENG_PRINCIPAL)
wm_c = ler_backup(WM_ENG_CALOR)
log("  Backup web map principal:", "ok" if wm_p else "não encontrado (uso padrão)")
log("  Backup web map de calor:", "ok" if wm_c else "não encontrado (uso padrão)")

def achar_layer(wm, item_id):
    for ol in (wm.get("operationalLayers") or []):
        if ol.get("itemId") == item_id:
            return ol
    return None

def drawing(wm, item_id):
    ol = achar_layer(wm, item_id) or {}
    return copy.deepcopy((ol.get("layerDefinition") or {}).get("drawingInfo"))

L, P, H = SRC["linhas"], SRC["pontos"], SRC["horarios"]
fs_bairros = gis_apuc.content.get(APUC_BAIRROS)
fs_eixos   = gis_apuc.content.get(APUC_EIXOS)

ARCADE_HORARIOS = """
var tab = FeatureSetByPortalItem(Portal('%s'), '%s', 0, ['id_rota','dia_tipo','ponto','horario','sentido'], false);
var rota = $feature.id_rota;
var hs = Filter(tab, 'id_rota = @rota');
if (Count(hs) == 0) {
  return { type: 'text', text: '<p style="color:#888;margin:4px 0">Sem horários cadastrados para esta rota.</p>' };
}
var html = '<div style="font-family:Arial,sans-serif;font-size:13px">';
for (var d in Distinct(hs, 'dia_tipo')) {
  var dia = d.dia_tipo;
  html += '<div style="margin:10px 0 4px;font-weight:bold;color:#1A2942;border-bottom:2px solid #F5A623">' + dia + '</div><div>';
  var sub = OrderBy(Filter(hs, 'dia_tipo = @dia'), 'horario ASC');
  for (var h in sub) {
    html += '<span style="display:inline-block;margin:2px;padding:3px 8px;border-radius:4px;background:#1A2942;color:#fff;font-weight:bold">' + h.horario + '</span>';
  }
  html += '</div>';
}
html += '</div>';
return { type: 'text', text: html };
""" % (URL_APUC, H["novo_id"])

pop_linhas = {
    "title": "Linha {cod_linha} — {nome}",
    "fieldInfos": [{"fieldName": f, "label": lab, "visible": True} for f, lab in
                   [("cod_linha", "Linha"), ("nome", "Nome"), ("sentido", "Sentido"),
                    ("sentido_variante", "Variante"), ("observacao", "Observação")]],
    "expressionInfos": [{"name": "horarios", "title": "Horários", "expression": ARCADE_HORARIOS, "returnType": "dictionary"}],
    "popupElements": [
        {"type": "fields"},
        {"type": "expression", "expressionInfo": {"name": "horarios", "title": "Horários", "expression": ARCADE_HORARIOS, "returnType": "dictionary"}},
    ],
}
pop_pontos = {"title": "Ponto de ônibus", "fieldInfos": [{"fieldName": "name", "label": "Ponto", "visible": True}],
              "popupElements": [{"type": "fields"}]}

calor_padrao = {"renderer": {"type": "heatmap", "blurRadius": 10, "maxPixelIntensity": 100, "minPixelIntensity": 0,
    "colorStops": [{"ratio": 0, "color": [133, 193, 200, 0]}, {"ratio": 0.01, "color": [144, 161, 190, 0]},
                   {"ratio": 0.0925, "color": [144, 161, 190, 130]}, {"ratio": 0.17, "color": [162, 98, 165, 200]},
                   {"ratio": 0.32, "color": [209, 52, 90, 255]}, {"ratio": 0.6, "color": [255, 98, 0, 255]},
                   {"ratio": 0.8, "color": [255, 200, 0, 255]}, {"ratio": 1, "color": [255, 255, 0, 255]}]}}
calor = drawing(wm_c, P["item"])
if not calor or (calor.get("renderer") or {}).get("type") != "heatmap":
    calor = calor_padrao

def lyr(titulo, url, item_id, visivel, drawinfo=None, popup=None, popup_on=True):
    d = {"id": re.sub(r"\W", "_", titulo) + "_" + item_id[:6], "title": titulo, "url": url, "itemId": item_id,
         "layerType": "ArcGISFeatureLayer", "visibility": visivel, "opacity": 1, "disablePopup": not popup_on}
    if drawinfo: d["layerDefinition"] = {"drawingInfo": drawinfo}
    if popup: d["popupInfo"] = popup
    return d

operacionais = [
    lyr("Bairros PMAPU", fs_bairros.url.rstrip("/") + "/0", APUC_BAIRROS, False),
    lyr("Eixos das Vias PMAPU", fs_eixos.url.rstrip("/") + "/0", APUC_EIXOS, False),
    lyr("Linhas de ônibus", L["novo_url"], L["novo_id"], True, drawing(wm_p, L["item"]), pop_linhas),
    lyr("Concentração de pontos de ônibus", P["novo_url"], P["novo_id"], False, calor, None, False),
    lyr("Pontos de ônibus", P["novo_url"], P["novo_id"], True, drawing(wm_p, P["item"]), pop_pontos),
]
tabelas = [{"id": "Horarios_" + H["novo_id"][:6], "title": "Horários de ônibus", "url": H["novo_url"], "itemId": H["novo_id"]}]

base = copy.deepcopy(wm_p.get("baseMap")) if wm_p.get("baseMap") else None
if not base or "engemap" in json.dumps(base).lower():
    base = {"title": "Imagens", "baseMapLayers": [{"id": "World_Imagery", "layerType": "ArcGISTiledMapServiceLayer",
            "url": "https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
            "visibility": True, "opacity": 1, "title": "World Imagery"}]}
wm_json = {"operationalLayers": operacionais, "tables": tabelas, "baseMap": base,
           "spatialReference": wm_p.get("spatialReference") or {"wkid": 102100, "latestWkid": 3857},
           "version": "2.27", "authoringApp": "GeoAPUC notebook", "authoringAppVersion": "1"}
if wm_p.get("initialState"):
    wm_json["initialState"] = wm_p["initialState"]

existente = [i for i in itens_na_pasta() if i.type == "Web Map" and i.title == WEBMAP_TITULO]
props = {"title": WEBMAP_TITULO, "type": "Web Map", "text": json.dumps(wm_json),
         "tags": "mobilidade, ônibus, transporte coletivo, Apucarana, GeoAPUC",
         "snippet": "Linhas, pontos e horários do transporte coletivo de Apucarana — base do app de Mobilidade Urbana (ExB)."}
if existente:
    wm_item = existente[0]
    wm_item.update(item_properties={"text": json.dumps(wm_json)})
    log("  Web map já existia — conteúdo atualizado:", wm_item.id)
else:
    wm_item = adicionar(props)
    log("  Web map criado:", wm_item.id)
compartilhar(wm_item)
LINKS.insert(0, ("Web map Mobilidade Urbana (Map Viewer)", f"{URL_APUC}/apps/mapviewer/index.html?webmap={wm_item.id}"))
LINKS.insert(1, ("Item do web map", f"{URL_APUC}/home/item.html?id={wm_item.id}"))
log("  Camadas no mapa: " + " | ".join(o["title"] for o in operacionais) + " | tabela: Horários de ônibus")
log("✔ CÉLULA 5 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 6 — IDs para o ExB + links =====
log("\n▶ CÉLULA 6 — Registro dos IDs e links: iniciando...")
ids = {"portal": URL_APUC, "pasta": PASTA_PORTAL, "webmap": wm_item.id,
       "linhas": {"item": L["novo_id"], "url": L["novo_url"], "n": L["n_destino"]},
       "pontos": {"item": P["novo_id"], "url": P["novo_url"], "n": P["n_destino"]},
       "horarios": {"item": H["novo_id"], "url": H["novo_url"], "n": H["n_destino"]},
       "bairros": APUC_BAIRROS, "eixos": APUC_EIXOS, "survey123": None}
arq_ids = os.path.join(PASTA_SAIDA, "ids_mobilidade_apucarana.json")
with open(arq_ids, "w", encoding="utf-8") as f:
    json.dump(ids, f, ensure_ascii=False, indent=2)
log("  IDs gravados em:", arq_ids)
log(json.dumps(ids, ensure_ascii=False, indent=2))
log("\n  ===== LINKS =====")
for nome, url in LINKS:
    log(f"   {nome}\n     {url}")
log("\n  Confira no Map Viewer: clique numa linha → o pop-up deve mostrar os horários agrupados por dia.")
log(f"  Saída completa: {ARQ_TXT}")
log("\n✔ CÉLULA 6 concluída com sucesso — publicação finalizada.")
